# AlpCAN — CXR Pipeline: ResNet-50 (512×512) Baseline

**Amaç:** NIH ChestX-ray14 tam veri seti üzerinde TorchXRayVision ResNet-50 (512×512) performansını değerlendirmek. AlpCAN CXR Pipeline Agent 7C (X-Raydar proxy) baseline.

**İçerik:**
1. Veri seti yükleme — NIH ChestX-ray14
2. TorchXRayVision ResNet-50 (512×512) model yükleme
3. Tam dataset inference (112K+ görüntü)
4. AUC-ROC performans + DenseNet-121 karşılaştırma
5. Threshold analizi (Nodule/Mass odaklı)
6. Multi-model ensemble (ResNet-50 + DenseNet-121) analizi
7. Çözünürlük etkisi analizi (224 vs 512)

**Dataset:** `nih-chest-xrays/data` (45GB, orijinal çözünürlük)

**Model:** resnet50-res512-all — 512×512 giriş çözünürlüğü

**GPU:** Kaggle T4 16GB

In [ ]:
# Kutuphaneleri kur
!pip install -q torchxrayvision scikit-learn

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

import torch
import torchxrayvision as xrv
from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_curve, average_precision_score

print(f"PyTorch: {torch.__version__}")
print(f"TorchXRayVision: {xrv.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"CUDA: {torch.cuda.is_available()}")

---
## 1. Veri Seti Yükleme — NIH ChestX-ray14

In [ ]:
# Dataset yolunu bul
INPUT_DIR = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working")

# Data_Entry CSV
csv_candidates = list(INPUT_DIR.rglob("Data_Entry*.csv"))
if not csv_candidates:
    raise FileNotFoundError("Data_Entry CSV bulunamadi! Dataset eklenmi\u015f mi?")
LABELS_CSV = csv_candidates[0]

# Goruntu klasorleri (images_001 - images_012)
all_img_dirs = sorted(INPUT_DIR.rglob("images"))
IMG_DIRS = [d for d in all_img_dirs if d.is_dir() and list(d.glob('*.png'))]

if not IMG_DIRS:
    all_pngs = list(INPUT_DIR.rglob("*.png"))
    if all_pngs:
        IMG_DIRS = list(set(p.parent for p in all_pngs[:10]))

print(f"Labels CSV: {LABELS_CSV}")
print(f"Image directories: {len(IMG_DIRS)}")
for d in IMG_DIRS[:5]:
    n_files = len(list(d.glob('*.png')))
    print(f"  {d}: {n_files:,} PNG")

In [ ]:
# Tum goruntu yollarini index'le
image_index = {}
for img_dir in IMG_DIRS:
    for png in img_dir.glob('*.png'):
        image_index[png.name] = png

print(f"Toplam indekslenen goruntu: {len(image_index):,}")

In [ ]:
# Labels CSV'yi oku
df = pd.read_csv(LABELS_CSV)
print(f"CSV kayit sayisi: {len(df):,}")
print(f"Hasta sayisi: {df['Patient ID'].nunique():,}")
print(f"\nSutunlar: {list(df.columns)}")
print(f"\nOrnek etiketler:")
print(df['Finding Labels'].value_counts().head(10))
df.head()

In [ ]:
# 14 patoloji etiketlerini binary sutunlara donustur
ALL_LABELS = [
    'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema',
    'Effusion', 'Emphysema', 'Fibrosis', 'Hernia',
    'Infiltration', 'Mass', 'Nodule', 'Pleural_Thickening',
    'Pneumonia', 'Pneumothorax'
]

for label in ALL_LABELS:
    df[label] = df['Finding Labels'].apply(lambda x: 1 if label in str(x) else 0)

df['No Finding'] = df['Finding Labels'].apply(lambda x: 1 if x == 'No Finding' else 0)

# Tam dagilim
print("=" * 65)
print("NIH ChestX-ray14 \u2014 Patoloji Dagilimi (TAM DATASET)")
print("=" * 65)

label_counts = df[ALL_LABELS].sum().sort_values(ascending=False)
for label, count in label_counts.items():
    pct = count / len(df) * 100
    marker = " \u2605 AlpCAN" if label in ('Nodule', 'Mass') else ""
    print(f"  {label:>20s}: {count:>6,} ({pct:>5.1f}%){marker}")

no_finding = df['No Finding'].sum()
print(f"  {'No Finding':>20s}: {no_finding:>6,} ({no_finding/len(df)*100:>5.1f}%)")
print(f"\n  Toplam goruntu: {len(df):,}")
print(f"  Toplam hasta:   {df['Patient ID'].nunique():,}")
print(f"  Multi-label:    {(df[ALL_LABELS].sum(axis=1) > 1).sum():,} goruntu birden fazla patoloji iceriyor")

---
## 2. TorchXRayVision ResNet-50 (512×512) Model Yükleme

In [ ]:
# TorchXRayVision ResNet-50 — 512x512 giris cozunurlugu, tum public CXR dataset'lerinden egitilmis
model = xrv.models.ResNet(weights="resnet50-res512-all")
model.eval()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

print(f"Model: ResNet-50 (resnet50-res512-all)")
print(f"Giris cozunurlugu: 512x512")
print(f"Device: {device}")

print(f"\nModel cikis etiketleri ({len(model.pathologies)}):")
for i, p in enumerate(model.pathologies):
    print(f"  [{i:>2}] {p}")

# Model parametreleri
n_params = sum(p.numel() for p in model.parameters())
print(f"\nParametre sayisi: {n_params:,} ({n_params/1e6:.1f}M)")

In [ ]:
# NIH <-> TorchXRayVision etiket eslestirmesi
TXRV_LABELS = list(model.pathologies)

LABEL_MAP = {}
for nih_label in ALL_LABELS:
    for txrv_label in TXRV_LABELS:
        if nih_label.lower().replace('_', '') == txrv_label.lower().replace('_', ''):
            LABEL_MAP[nih_label] = txrv_label
            break

print("NIH \u2192 TorchXRayVision Etiket Eslestirmesi:")
print("-" * 50)
for nih, txrv in sorted(LABEL_MAP.items()):
    txrv_idx = TXRV_LABELS.index(txrv)
    print(f"  {nih:>20s} \u2192 [{txrv_idx:>2}] {txrv}")

unmatched = set(ALL_LABELS) - set(LABEL_MAP.keys())
print(f"\n  Eslesen: {len(LABEL_MAP)}/{len(ALL_LABELS)}")
if unmatched:
    print(f"  Eslesmeyen: {unmatched}")

---
## 3. Tam Dataset Inference (512×512)

In [ ]:
def load_and_preprocess(image_path):
    """CXR goruntusunu TorchXRayVision ResNet-50 formatina cevir.
    
    - Grayscale oku
    - 512x512 resize (ResNet-50 icin)
    - [0,255] -> [-1024, 1024] normalize (TorchXRayVision convention)
    """
    img = Image.open(image_path).convert('L')
    img = img.resize((512, 512), Image.LANCZOS)  # 512 for ResNet-50
    img = np.array(img, dtype=np.float32)
    
    # TorchXRayVision normalizasyonu
    img = xrv.datasets.normalize(img, 255)
    
    # (H,W) -> (1,H,W)
    img = img[np.newaxis, :, :]
    return torch.from_numpy(img)


# Hizli test
test_name = list(image_index.keys())[0]
test_img = load_and_preprocess(image_index[test_name])
print(f"Test: {test_name}")
print(f"Shape: {test_img.shape}, Range: [{test_img.min():.1f}, {test_img.max():.1f}]")

with torch.no_grad():
    out = model(test_img.unsqueeze(0).to(device)).cpu().numpy()[0]
print(f"Output shape: {out.shape}")
for i, (label, score) in enumerate(zip(model.pathologies, out)):
    if score > 0.3:
        print(f"  [{i}] {label}: {score:.4f}")

In [ ]:
# Tam dataset inference
# 112K goruntu — 512x512 cozunurluk nedeniyle daha kucuk batch
BATCH_SIZE = 32  # 512x512 icin daha kucuk batch (GPU bellek)

# Sadece mevcut goruntuleri filtrele
df['exists'] = df['Image Index'].apply(lambda x: x in image_index)
eval_df = df[df['exists']].reset_index(drop=True)
print(f"Mevcut goruntu: {len(eval_df):,} / {len(df):,}")

all_predictions = []
all_image_names = []
skipped = 0
start_time = time.time()

n_batches = (len(eval_df) + BATCH_SIZE - 1) // BATCH_SIZE
print(f"\nInference basliyor: {len(eval_df):,} goruntu, {n_batches:,} batch (batch_size={BATCH_SIZE})")
print(f"Cozunurluk: 512x512 (ResNet-50)")
print("-" * 60)

for batch_start in range(0, len(eval_df), BATCH_SIZE):
    batch_df = eval_df.iloc[batch_start:batch_start + BATCH_SIZE]
    batch_tensors = []
    batch_names = []

    for _, row in batch_df.iterrows():
        img_name = row['Image Index']
        img_path = image_index.get(img_name)
        if img_path is None:
            skipped += 1
            continue
        try:
            tensor = load_and_preprocess(img_path)
            batch_tensors.append(tensor)
            batch_names.append(img_name)
        except Exception:
            skipped += 1
            continue

    if not batch_tensors:
        continue

    batch_input = torch.stack(batch_tensors).to(device)
    with torch.no_grad():
        batch_output = model(batch_input)
        batch_preds = batch_output.cpu().numpy()

    all_predictions.append(batch_preds)
    all_image_names.extend(batch_names)

    processed = len(all_image_names)
    if processed % 5000 < BATCH_SIZE or batch_start + BATCH_SIZE >= len(eval_df):
        elapsed = time.time() - start_time
        speed = processed / elapsed
        eta = (len(eval_df) - processed) / speed if speed > 0 else 0
        print(f"  {processed:>7,}/{len(eval_df):,} ({processed/len(eval_df)*100:>5.1f}%) "
              f"| {speed:.0f} img/s | ETA: {eta/60:.1f} min")

# Birlestir
predictions = np.concatenate(all_predictions, axis=0)
elapsed = time.time() - start_time

print(f"\n{'=' * 60}")
print(f"Inference tamamlandi!")
print(f"   Islenen: {predictions.shape[0]:,} goruntu")
print(f"   Atlanan: {skipped}")
print(f"   Sure: {elapsed/60:.1f} dakika ({predictions.shape[0]/elapsed:.0f} img/s)")
print(f"   Prediction shape: {predictions.shape}")

---
## 4. AUC-ROC Performans + DenseNet-121 Karşılaştırma

In [ ]:
# Ground truth al
eval_processed = eval_df[eval_df['Image Index'].isin(all_image_names)].reset_index(drop=True)
assert len(eval_processed) == len(predictions), "Boyut uyusmazligi!"

# AUC-ROC + AP (Average Precision) hesapla
print("=" * 70)
print("TorchXRayVision ResNet-50 (512x512) \u2014 NIH ChestX-ray14 Performansi")
print(f"N = {len(predictions):,} goruntu")
print("=" * 70)
print(f"{'Patoloji':>20s} | {'AUC-ROC':>8s} | {'AP':>8s} | {'N_pos':>7s} | {'Prevalans':>9s}")
print("-" * 70)

auc_results = {}
ap_results = {}

for nih_label, txrv_label in LABEL_MAP.items():
    txrv_idx = TXRV_LABELS.index(txrv_label)
    y_true = eval_processed[nih_label].values
    y_score = predictions[:, txrv_idx]

    n_pos = y_true.sum()
    prevalence = n_pos / len(y_true) * 100

    if n_pos > 0 and n_pos < len(y_true):
        auc = roc_auc_score(y_true, y_score)
        ap = average_precision_score(y_true, y_score)
        auc_results[nih_label] = auc
        ap_results[nih_label] = ap
        marker = " \u2605" if nih_label in ('Nodule', 'Mass') else ""
        print(f"  {nih_label:>20s} | {auc:>8.4f} | {ap:>8.4f} | {n_pos:>7,} | {prevalence:>8.1f}%{marker}")
    else:
        print(f"  {nih_label:>20s} | {'N/A':>8s} | {'N/A':>8s} | {n_pos:>7,} | {prevalence:>8.1f}%")

print("-" * 70)
if auc_results:
    mean_auc = np.mean(list(auc_results.values()))
    mean_ap = np.mean(list(ap_results.values()))
    print(f"  {'Ortalama':>20s} | {mean_auc:>8.4f} | {mean_ap:>8.4f} |")

    cancer_labels = [l for l in ('Nodule', 'Mass') if l in auc_results]
    if cancer_labels:
        cancer_auc = np.mean([auc_results[l] for l in cancer_labels])
        cancer_ap = np.mean([ap_results[l] for l in cancer_labels])
        print(f"  {'Nodule+Mass':>20s} | {cancer_auc:>8.4f} | {cancer_ap:>8.4f} | \u2190 AlpCAN odak")

In [ ]:
# DenseNet-121 sonuclari ile karsilastirma
# Onceki notebook ciktisi varsa yukle
DENSENET_AUC = {}
densenet_csv_path = OUTPUT_DIR / 'cxr_auc_scores.csv'

# Kaggle kernel source'dan da dene
kernel_source_paths = [
    OUTPUT_DIR / 'cxr_auc_scores.csv',
    INPUT_DIR / 'alpcan-cxr-pipeline-torchxrayvision-baseline' / 'cxr_auc_scores.csv',
    Path('/kaggle/input/alpcan-cxr-pipeline-torchxrayvision-baseline/cxr_auc_scores.csv'),
]

densenet_loaded = False
for p in kernel_source_paths:
    if p.exists():
        densenet_auc_df = pd.read_csv(p)
        for _, row in densenet_auc_df.iterrows():
            DENSENET_AUC[row['pathology']] = row['auc_roc']
        print(f"DenseNet-121 AUC sonuclari yuklendi: {p}")
        print(f"  {len(DENSENET_AUC)} patoloji")
        densenet_loaded = True
        break

if not densenet_loaded:
    print("DenseNet-121 AUC sonuclari bulunamadi.")
    print("Notebook 02 ciktisi mevcut degil - karsilastirma Wang 2017 ile yapilacak.")

# Wang et al. 2017 orijinal sonuclari
WANG_AUC = {
    'Atelectasis': 0.716, 'Cardiomegaly': 0.814, 'Effusion': 0.784,
    'Infiltration': 0.609, 'Mass': 0.706, 'Nodule': 0.671,
    'Pneumonia': 0.633, 'Pneumothorax': 0.806, 'Consolidation': 0.708,
    'Edema': 0.805, 'Emphysema': 0.833, 'Fibrosis': 0.786,
    'Pleural_Thickening': 0.684, 'Hernia': 0.872
}

In [ ]:
# Uclu karsilastirma tablosu: ResNet-50 vs DenseNet-121 vs Wang 2017
print("\n" + "=" * 80)
print("ResNet-50 (512) vs DenseNet-121 (224) vs Wang et al. 2017 (CVPR)")
print("=" * 80)

if DENSENET_AUC:
    print(f"{'Patoloji':>20s} | {'ResNet50':>8s} | {'DenseNet':>8s} | {'Wang':>8s} | {'R50-D121':>8s} | {'R50-Wang':>8s}")
    print("-" * 80)
    
    r50_vs_d121 = []
    r50_vs_wang = []
    for label in sorted(auc_results.keys()):
        r50 = auc_results[label]
        d121 = DENSENET_AUC.get(label, None)
        wang = WANG_AUC.get(label, None)
        
        diff_d121 = f"{r50 - d121:>+7.4f}" if d121 is not None else "    N/A"
        diff_wang = f"{r50 - wang:>+7.4f}" if wang is not None else "    N/A"
        d121_str = f"{d121:>8.4f}" if d121 is not None else "     N/A"
        wang_str = f"{wang:>8.4f}" if wang is not None else "     N/A"
        
        if d121 is not None:
            r50_vs_d121.append(r50 - d121)
        if wang is not None:
            r50_vs_wang.append(r50 - wang)
        
        marker = " *" if label in ('Nodule', 'Mass') else ""
        print(f"  {label:>20s} | {r50:>8.4f} | {d121_str} | {wang_str} | {diff_d121} | {diff_wang}{marker}")
    
    print("-" * 80)
    if r50_vs_d121:
        print(f"  {'Ort. fark':>20s} | {'':>8s} | {'':>8s} | {'':>8s} | {np.mean(r50_vs_d121):>+7.4f} | {np.mean(r50_vs_wang):>+7.4f}")
else:
    print(f"{'Patoloji':>20s} | {'ResNet50':>8s} | {'Wang':>8s} | {'Fark':>8s}")
    print("-" * 60)
    
    improvements = []
    for label in sorted(auc_results.keys()):
        ours = auc_results[label]
        wang = WANG_AUC.get(label, 0)
        diff = ours - wang
        improvements.append(diff)
        arrow = "\u2191" if diff > 0 else "\u2193"
        print(f"  {label:>20s} | {ours:>8.4f} | {wang:>8.4f} | {diff:>+7.4f} {arrow}")
    
    print("-" * 60)
    print(f"  {'Ortalama fark':>20s} | {'':>8s} | {'':>8s} | {np.mean(improvements):>+7.4f}")

In [ ]:
# Performans gorsellestirmesi
fig, axes = plt.subplots(1, 3, figsize=(22, 7))

# 1. AUC karsilastirma
labels_sorted = sorted(auc_results.keys(), key=lambda x: auc_results[x], reverse=True)
r50_vals = [auc_results[l] for l in labels_sorted]
wang_vals = [WANG_AUC.get(l, 0) for l in labels_sorted]

y_pos = np.arange(len(labels_sorted))
bar_width = 0.25 if DENSENET_AUC else 0.3

if DENSENET_AUC:
    d121_vals = [DENSENET_AUC.get(l, 0) for l in labels_sorted]
    axes[0].barh(y_pos + bar_width, wang_vals, bar_width, label='Wang 2017', color='#bdc3c7', alpha=0.8)
    axes[0].barh(y_pos, d121_vals, bar_width, label='DenseNet-121 (224)', color='#3498db')
    axes[0].barh(y_pos - bar_width, r50_vals, bar_width, label='ResNet-50 (512)', color='#e74c3c')
else:
    axes[0].barh(y_pos + 0.15, wang_vals, 0.3, label='Wang 2017', color='#bdc3c7', alpha=0.8)
    axes[0].barh(y_pos - 0.15, r50_vals, 0.3, label='ResNet-50 (512)', color='#e74c3c')

# Nodule ve Mass'i vurgula
for i, l in enumerate(labels_sorted):
    if l in ('Nodule', 'Mass'):
        offset = -bar_width if DENSENET_AUC else -0.15
        axes[0].barh(i + offset, auc_results[l], bar_width if DENSENET_AUC else 0.3, color='#c0392b', edgecolor='black', linewidth=1.5)

axes[0].set_yticks(y_pos)
axes[0].set_yticklabels(labels_sorted)
axes[0].set_xlim(0.5, 1.0)
axes[0].set_title('AUC-ROC Karsilastirma', fontsize=13, fontweight='bold')
axes[0].set_xlabel('AUC-ROC')
axes[0].legend(loc='lower right', fontsize=9)
axes[0].axvline(x=0.7, color='gray', linestyle='--', alpha=0.3)
axes[0].axvline(x=0.8, color='gray', linestyle='--', alpha=0.3)
axes[0].invert_yaxis()

# 2. AlpCAN odak — Nodule & Mass ROC
for label in ('Nodule', 'Mass', 'Consolidation', 'Infiltration'):
    if label in LABEL_MAP:
        txrv_idx = TXRV_LABELS.index(LABEL_MAP[label])
        y_true = eval_processed[label].values
        y_score = predictions[:, txrv_idx]
        if y_true.sum() > 0:
            fpr, tpr, _ = roc_curve(y_true, y_score)
            auc = auc_results[label]
            lw = 3 if label in ('Nodule', 'Mass') else 1.5
            axes[1].plot(fpr, tpr, label=f"{label} (AUC={auc:.3f})", linewidth=lw)

axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.3)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Egrileri \u2014 AlpCAN Odak Patolojileri', fontsize=13, fontweight='bold')
axes[1].legend(loc='lower right')
axes[1].grid(True, alpha=0.2)

# 3. Precision-Recall egrileri
for label in ('Nodule', 'Mass'):
    if label in LABEL_MAP:
        txrv_idx = TXRV_LABELS.index(LABEL_MAP[label])
        y_true = eval_processed[label].values
        y_score = predictions[:, txrv_idx]
        if y_true.sum() > 0:
            prec, rec, _ = precision_recall_curve(y_true, y_score)
            ap = ap_results[label]
            axes[2].plot(rec, prec, label=f"{label} (AP={ap:.3f})", linewidth=2)
            prevalence = y_true.mean()
            axes[2].axhline(y=prevalence, color='gray', linestyle=':', alpha=0.3)

axes[2].set_xlabel('Recall')
axes[2].set_ylabel('Precision')
axes[2].set_title('Precision-Recall \u2014 Nodule & Mass', fontsize=13, fontweight='bold')
axes[2].legend(loc='upper right')
axes[2].grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'cxr_resnet50_performance.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: cxr_resnet50_performance.png")

---
## 5. Threshold Analizi (Nodule/Mass Odaklı)

In [ ]:
# Nodule ve Mass icin detayli threshold analizi
fig, axes = plt.subplots(2, 3, figsize=(20, 12))

optimal_thresholds = {}

for idx, label in enumerate(['Nodule', 'Mass']):
    if label not in LABEL_MAP:
        continue

    txrv_idx = TXRV_LABELS.index(LABEL_MAP[label])
    y_true = eval_processed[label].values
    y_score = predictions[:, txrv_idx]

    # 1. Score dagilimi
    ax = axes[idx, 0]
    ax.hist(y_score[y_true == 0], bins=80, alpha=0.6, label=f'Negatif (n={sum(y_true==0):,})',
            color='#3498db', density=True)
    ax.hist(y_score[y_true == 1], bins=80, alpha=0.6, label=f'Pozitif (n={sum(y_true==1):,})',
            color='#e74c3c', density=True)
    ax.set_title(f'{label} \u2014 Tahmin Skoru Dagilimi (ResNet-50)', fontweight='bold')
    ax.set_xlabel('Skor')
    ax.set_ylabel('Yogunluk')
    ax.legend()

    # 2. Sensitivity / Specificity vs Threshold
    ax = axes[idx, 1]
    thresholds = np.arange(0.0, 1.0, 0.005)
    sensitivities, specificities, ppvs, npvs = [], [], [], []

    for th in thresholds:
        y_pred = (y_score >= th).astype(int)
        tp = ((y_pred == 1) & (y_true == 1)).sum()
        tn = ((y_pred == 0) & (y_true == 0)).sum()
        fp = ((y_pred == 1) & (y_true == 0)).sum()
        fn = ((y_pred == 0) & (y_true == 1)).sum()

        sens = tp / (tp + fn) if (tp + fn) > 0 else 0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0
        ppv = tp / (tp + fp) if (tp + fp) > 0 else 0
        npv = tn / (tn + fn) if (tn + fn) > 0 else 0

        sensitivities.append(sens)
        specificities.append(spec)
        ppvs.append(ppv)
        npvs.append(npv)

    ax.plot(thresholds, sensitivities, label='Sensitivity', color='#e74c3c', linewidth=2)
    ax.plot(thresholds, specificities, label='Specificity', color='#3498db', linewidth=2)
    ax.plot(thresholds, ppvs, label='PPV', color='#27ae60', linewidth=1, linestyle='--')
    ax.plot(thresholds, npvs, label='NPV', color='#8e44ad', linewidth=1, linestyle='--')

    # Optimal threshold (Youden's J)
    j_scores = np.array(sensitivities) + np.array(specificities) - 1
    opt_idx = np.argmax(j_scores)
    opt_th = thresholds[opt_idx]
    optimal_thresholds[label] = opt_th

    # Yuksek sensitivity threshold (tarama icin >=0.90 sensitivity hedef)
    high_sens_idx = None
    for i, s in enumerate(sensitivities):
        if s >= 0.90:
            high_sens_idx = i

    ax.axvline(x=opt_th, color='#f39c12', linestyle='--', alpha=0.8,
               label=f'Youden opt: {opt_th:.3f}')
    if high_sens_idx is not None:
        hs_th = thresholds[high_sens_idx]
        ax.axvline(x=hs_th, color='#e74c3c', linestyle=':', alpha=0.8,
                   label=f'Sens>=0.90: {hs_th:.3f}')

    ax.set_title(f'{label} \u2014 Threshold Analizi (ResNet-50)', fontweight='bold')
    ax.set_xlabel('Threshold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.2)

    # 3. F1 ve Matthews Correlation
    ax = axes[idx, 2]
    f1_scores = []
    for s, p in zip(sensitivities, ppvs):
        f1 = 2 * s * p / (s + p) if (s + p) > 0 else 0
        f1_scores.append(f1)

    ax.plot(thresholds, f1_scores, label='F1 Score', color='#2c3e50', linewidth=2)
    ax.plot(thresholds, j_scores, label="Youden's J", color='#e67e22', linewidth=2)
    f1_opt = thresholds[np.argmax(f1_scores)]
    ax.axvline(x=f1_opt, color='#2c3e50', linestyle='--', alpha=0.5,
               label=f'F1 opt: {f1_opt:.3f}')
    ax.set_title(f'{label} \u2014 F1 & Youden J (ResNet-50)', fontweight='bold')
    ax.set_xlabel('Threshold')
    ax.legend()
    ax.grid(True, alpha=0.2)

    print(f"\n{label}:")
    print(f"  Youden optimal threshold: {opt_th:.3f}")
    print(f"    Sensitivity: {sensitivities[opt_idx]:.3f}, Specificity: {specificities[opt_idx]:.3f}")
    print(f"    PPV: {ppvs[opt_idx]:.3f}, NPV: {npvs[opt_idx]:.3f}")
    if high_sens_idx is not None:
        print(f"  High-sensitivity threshold (>=0.90): {thresholds[high_sens_idx]:.3f}")
        print(f"    Sensitivity: {sensitivities[high_sens_idx]:.3f}, Specificity: {specificities[high_sens_idx]:.3f}")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'cxr_resnet50_threshold.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nSaved: cxr_resnet50_threshold.png")

---
## 6. Multi-Model Ensemble (ResNet-50 + DenseNet-121) Analizi

In [ ]:
# DenseNet-121 tahminlerini yukle (varsa)
densenet_preds = None
densenet_preds_paths = [
    OUTPUT_DIR / 'cxr_predictions_full.csv',
    INPUT_DIR / 'alpcan-cxr-pipeline-torchxrayvision-baseline' / 'cxr_predictions_full.csv',
    Path('/kaggle/input/alpcan-cxr-pipeline-torchxrayvision-baseline/cxr_predictions_full.csv'),
]

densenet_preds_df = None
for p in densenet_preds_paths:
    if p.exists():
        densenet_preds_df = pd.read_csv(p)
        print(f"DenseNet-121 tahminleri yuklendi: {p}")
        print(f"  Shape: {densenet_preds_df.shape}")
        break

if densenet_preds_df is None:
    print("DenseNet-121 tahminleri bulunamadi.")
    print("Ensemble analizi icin Notebook 02'yi calistirin ve kernel_sources'a ekleyin.")
    print("\nAlternatif: ResNet-50 kendi ici ensemble simulasyonu yapilacak.")

In [ ]:
# Multi-model agreement analizi
if densenet_preds_df is not None:
    # Ortak goruntuleri bul
    resnet_names = set(all_image_names)
    densenet_names = set(densenet_preds_df['image_name'].values)
    common_names = sorted(resnet_names & densenet_names)
    print(f"Ortak goruntu sayisi: {len(common_names):,}")
    
    # Nodule ve Mass icin agreement analizi
    for label in ['Nodule', 'Mass']:
        if label not in LABEL_MAP:
            continue
        
        txrv_label = LABEL_MAP[label]
        txrv_idx = TXRV_LABELS.index(txrv_label)
        
        # ResNet-50 skorlari
        r50_df = pd.DataFrame({'image_name': all_image_names, 'r50_score': predictions[:, txrv_idx]})
        
        # DenseNet-121 skorlari
        d121_col = f'pred_{txrv_label}'
        if d121_col not in densenet_preds_df.columns:
            # Alternatif isim dene
            d121_col_candidates = [c for c in densenet_preds_df.columns if txrv_label.lower() in c.lower()]
            d121_col = d121_col_candidates[0] if d121_col_candidates else None
        
        if d121_col and d121_col in densenet_preds_df.columns:
            d121_df = densenet_preds_df[['image_name', d121_col]].rename(columns={d121_col: 'd121_score'})
            merged = r50_df.merge(d121_df, on='image_name', how='inner')
            
            # Ground truth ekle
            gt_df = eval_processed[['Image Index', label]].rename(columns={'Image Index': 'image_name'})
            merged = merged.merge(gt_df, on='image_name', how='inner')
            
            # Threshold uygula
            r50_th = optimal_thresholds.get(label, 0.4)
            d121_th = r50_th  # Ayni threshold kullan (basitlik icin)
            
            merged['r50_pred'] = (merged['r50_score'] >= r50_th).astype(int)
            merged['d121_pred'] = (merged['d121_score'] >= d121_th).astype(int)
            
            # Agreement matrisi
            both_pos = ((merged['r50_pred'] == 1) & (merged['d121_pred'] == 1)).sum()
            only_r50 = ((merged['r50_pred'] == 1) & (merged['d121_pred'] == 0)).sum()
            only_d121 = ((merged['r50_pred'] == 0) & (merged['d121_pred'] == 1)).sum()
            both_neg = ((merged['r50_pred'] == 0) & (merged['d121_pred'] == 0)).sum()
            
            agreement = (both_pos + both_neg) / len(merged) * 100
            
            print(f"\n{label} Agreement Analizi (threshold={r50_th:.3f}):")
            print(f"  Her iki model pozitif:  {both_pos:>7,}")
            print(f"  Sadece ResNet-50:       {only_r50:>7,}")
            print(f"  Sadece DenseNet-121:    {only_d121:>7,}")
            print(f"  Her iki model negatif:  {both_neg:>7,}")
            print(f"  Agreement:              {agreement:.1f}%")
            
            # Ensemble (ortalama) performansi
            ensemble_score = (merged['r50_score'] + merged['d121_score']) / 2
            y_true = merged[label].values
            if y_true.sum() > 0 and y_true.sum() < len(y_true):
                ensemble_auc = roc_auc_score(y_true, ensemble_score)
                r50_auc = roc_auc_score(y_true, merged['r50_score'])
                d121_auc = roc_auc_score(y_true, merged['d121_score'])
                print(f"\n  AUC Karsilastirma:")
                print(f"    ResNet-50:           {r50_auc:.4f}")
                print(f"    DenseNet-121:        {d121_auc:.4f}")
                print(f"    Ensemble (ort.):     {ensemble_auc:.4f}")
                improvement = ensemble_auc - max(r50_auc, d121_auc)
                print(f"    Ensemble kazanimi:   {improvement:+.4f}")
else:
    # DenseNet-121 yoksa, ResNet-50 kendi ici ensemble simulasyonu
    print("\nResNet-50 tabanli ensemble simulasyonu (4 model proxy):")
    print("-" * 60)
    
    if 'Nodule' in LABEL_MAP and 'Mass' in LABEL_MAP:
        nodule_idx = TXRV_LABELS.index(LABEL_MAP['Nodule'])
        mass_idx = TXRV_LABELS.index(LABEL_MAP['Mass'])
        combined_scores = np.maximum(predictions[:, nodule_idx], predictions[:, mass_idx])
        
        # 4 model simulasyonu
        model_configs = {
            'Ark+ (Swin-L)':         {'threshold': optimal_thresholds.get('Nodule', 0.35) * 0.85},
            'ResNet-50 (512)':       {'threshold': optimal_thresholds.get('Nodule', 0.40)},
            'X-Raydar (ResNet+Tr)':  {'threshold': optimal_thresholds.get('Nodule', 0.40) * 1.10},
            'MedRAX (MedSAM2)':      {'threshold': optimal_thresholds.get('Nodule', 0.35) * 0.80},
        }
        
        votes = np.zeros(len(combined_scores), dtype=int)
        print("Model Konfigurasyonlari (Simule):")
        for name, cfg in model_configs.items():
            model_votes = (combined_scores >= cfg['threshold']).astype(int)
            votes += model_votes
            print(f"  {name:>25s}: threshold={cfg['threshold']:.3f}, pozitif={model_votes.sum():,}")
        
        # Karar dagilimi
        ENSEMBLE_DECISIONS = {
            4: {"priority": "YUKSEK ONCELIK", "action": "Acil BT onerisi"},
            3: {"priority": "ORTA-YUKSEK", "action": "2 hafta icinde BT"},
            2: {"priority": "ORTA", "action": "Radyolog degerlendirmesi"},
            1: {"priority": "DUSUK", "action": "Takip hatirlatici"},
            0: {"priority": "NORMAL", "action": "Rutin akis"},
        }
        
        print(f"\n{'=' * 60}")
        print(f"Ensemble Karar Dagilimi (N={len(votes):,})")
        print(f"{'=' * 60}")
        
        vote_counts = pd.Series(votes).value_counts().sort_index()
        for v in range(5):
            count = vote_counts.get(v, 0)
            pct = count / len(votes) * 100
            d = ENSEMBLE_DECISIONS[v]
            print(f"  {v}/4 oy: {d['priority']:>15s}: {count:>7,} ({pct:>5.1f}%) \u2014 {d['action']}")

In [ ]:
# Ensemble gorsellestirmesi
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

if densenet_preds_df is not None and 'Nodule' in LABEL_MAP:
    txrv_label = LABEL_MAP['Nodule']
    txrv_idx = TXRV_LABELS.index(txrv_label)
    r50_df = pd.DataFrame({'image_name': all_image_names, 'r50_score': predictions[:, txrv_idx]})
    
    d121_col = f'pred_{txrv_label}'
    if d121_col in densenet_preds_df.columns:
        d121_df = densenet_preds_df[['image_name', d121_col]].rename(columns={d121_col: 'd121_score'})
        merged = r50_df.merge(d121_df, on='image_name', how='inner')
        gt_df = eval_processed[['Image Index', 'Nodule']].rename(columns={'Image Index': 'image_name'})
        merged = merged.merge(gt_df, on='image_name', how='inner')
        
        # Scatter: R50 vs D121 skorlari
        pos_mask = merged['Nodule'] == 1
        neg_sample = merged[~pos_mask].sample(min(5000, (~pos_mask).sum()), random_state=42)
        
        axes[0].scatter(neg_sample['r50_score'], neg_sample['d121_score'],
                       alpha=0.1, s=5, c='#3498db', label='Negatif')
        axes[0].scatter(merged.loc[pos_mask, 'r50_score'], merged.loc[pos_mask, 'd121_score'],
                       alpha=0.4, s=15, c='#e74c3c', label='Pozitif')
        axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.3)
        axes[0].set_xlabel('ResNet-50 (512) Skor')
        axes[0].set_ylabel('DenseNet-121 (224) Skor')
        axes[0].set_title('Nodule: ResNet-50 vs DenseNet-121', fontweight='bold')
        axes[0].legend()
        axes[0].grid(True, alpha=0.2)
        
        # ROC karsilastirma
        y_true = merged['Nodule'].values
        if y_true.sum() > 0:
            for score_col, name, color in [
                ('r50_score', 'ResNet-50 (512)', '#e74c3c'),
                ('d121_score', 'DenseNet-121 (224)', '#3498db'),
            ]:
                fpr, tpr, _ = roc_curve(y_true, merged[score_col])
                auc_val = roc_auc_score(y_true, merged[score_col])
                axes[1].plot(fpr, tpr, label=f"{name} (AUC={auc_val:.4f})", color=color, linewidth=2)
            
            ensemble_score = (merged['r50_score'] + merged['d121_score']) / 2
            fpr, tpr, _ = roc_curve(y_true, ensemble_score)
            ens_auc = roc_auc_score(y_true, ensemble_score)
            axes[1].plot(fpr, tpr, label=f"Ensemble (AUC={ens_auc:.4f})",
                        color='#27ae60', linewidth=2.5, linestyle='--')
            
            axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.3)
            axes[1].set_xlabel('False Positive Rate')
            axes[1].set_ylabel('True Positive Rate')
            axes[1].set_title('Nodule ROC: Tek Model vs Ensemble', fontweight='bold')
            axes[1].legend(loc='lower right')
            axes[1].grid(True, alpha=0.2)
else:
    # DenseNet yoksa, simulasyon gorseli
    if 'Nodule' in LABEL_MAP:
        nodule_idx = TXRV_LABELS.index(LABEL_MAP['Nodule'])
        nodule_scores = predictions[:, nodule_idx]
        y_true = eval_processed['Nodule'].values
        
        # Score dagilimi pozitif vs negatif
        axes[0].hist(nodule_scores[y_true == 0], bins=80, alpha=0.5, density=True,
                    color='#3498db', label='Negatif')
        axes[0].hist(nodule_scores[y_true == 1], bins=80, alpha=0.5, density=True,
                    color='#e74c3c', label='Pozitif')
        if 'Nodule' in optimal_thresholds:
            axes[0].axvline(x=optimal_thresholds['Nodule'], color='orange', linestyle='--',
                           label=f'Threshold={optimal_thresholds["Nodule"]:.3f}')
        axes[0].set_title('Nodule Score Dagilimi (ResNet-50)', fontweight='bold')
        axes[0].set_xlabel('Score')
        axes[0].legend()
    
    # Ensemble simulasyon dagilimi
    if 'votes' in dir():
        vote_colors = ['#3498db', '#27ae60', '#f39c12', '#e67e22', '#e74c3c']
        vote_labels_viz = [f"{v}/4 oy" for v in range(5)]
        vote_vals = [vote_counts.get(v, 0) for v in range(5)]
        axes[1].barh(vote_labels_viz, vote_vals, color=vote_colors)
        axes[1].set_title('Ensemble Oylama Dagilimi', fontweight='bold')
        axes[1].set_xlabel('Goruntu Sayisi')
        axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'cxr_resnet50_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: cxr_resnet50_comparison.png")

---
## 7. Çözünürlük Etkisi Analizi (224 vs 512)

In [ ]:
# Cozunurluk etkisini olcmek icin ayni ResNet-50 modeli 224x224 ile de calistir
# Kucuk bir alt kumeyle (5000 goruntu)

RESOLUTION_SUBSET = 5000

def load_and_preprocess_224(image_path):
    """224x224 cozunurlukle on-isleme (karsilastirma icin)."""
    img = Image.open(image_path).convert('L')
    img = img.resize((224, 224), Image.LANCZOS)  # 224 for comparison
    img = np.array(img, dtype=np.float32)
    img = xrv.datasets.normalize(img, 255)
    img = img[np.newaxis, :, :]
    return torch.from_numpy(img)


# Rastgele alt kume sec (dengeli: pozitif+negatif)
np.random.seed(42)
subset_indices = np.random.choice(len(eval_processed), min(RESOLUTION_SUBSET, len(eval_processed)), replace=False)
subset_df = eval_processed.iloc[subset_indices].reset_index(drop=True)
print(f"Cozunurluk karsilastirma alt kumesi: {len(subset_df):,} goruntu")
print(f"  Nodule pozitif: {subset_df['Nodule'].sum():,}")
print(f"  Mass pozitif:   {subset_df['Mass'].sum():,}")

# 224x224 inference
preds_224 = []
preds_512 = []
valid_names_res = []

BATCH_SIZE_224 = 64  # 224x224 daha kucuk, batch buyutulebilir
start_time_res = time.time()

print(f"\n224x224 inference basliyor...")
for batch_start in range(0, len(subset_df), BATCH_SIZE_224):
    batch_df = subset_df.iloc[batch_start:batch_start + BATCH_SIZE_224]
    batch_224 = []
    batch_512 = []
    batch_names = []
    
    for _, row in batch_df.iterrows():
        img_name = row['Image Index']
        img_path = image_index.get(img_name)
        if img_path is None:
            continue
        try:
            t224 = load_and_preprocess_224(img_path)
            t512 = load_and_preprocess(img_path)
            batch_224.append(t224)
            batch_512.append(t512)
            batch_names.append(img_name)
        except Exception:
            continue
    
    if not batch_224:
        continue
    
    # 224x224 inference
    with torch.no_grad():
        out_224 = model(torch.stack(batch_224).to(device)).cpu().numpy()
        out_512 = model(torch.stack(batch_512).to(device)).cpu().numpy()
    
    preds_224.append(out_224)
    preds_512.append(out_512)
    valid_names_res.extend(batch_names)

preds_224 = np.concatenate(preds_224, axis=0)
preds_512 = np.concatenate(preds_512, axis=0)
elapsed_res = time.time() - start_time_res

print(f"Tamamlandi: {len(valid_names_res):,} goruntu, {elapsed_res:.1f}s")

In [ ]:
# Cozunurluk karsilastirma sonuclari
res_eval_df = subset_df[subset_df['Image Index'].isin(valid_names_res)].reset_index(drop=True)

print("=" * 70)
print("Cozunurluk Etkisi: ResNet-50 @ 224x224 vs 512x512")
print(f"N = {len(preds_224):,} goruntu")
print("=" * 70)
print(f"{'Patoloji':>20s} | {'224x224':>8s} | {'512x512':>8s} | {'Fark':>8s} | {'Etki':>10s}")
print("-" * 70)

res_comparison = {}
for nih_label, txrv_label in LABEL_MAP.items():
    txrv_idx = TXRV_LABELS.index(txrv_label)
    y_true = res_eval_df[nih_label].values
    
    if y_true.sum() > 10 and y_true.sum() < len(y_true) - 10:  # Yeterli ornek
        auc_224 = roc_auc_score(y_true, preds_224[:, txrv_idx])
        auc_512 = roc_auc_score(y_true, preds_512[:, txrv_idx])
        diff = auc_512 - auc_224
        
        if diff > 0.01:
            etki = "512 DAHA IYI"
        elif diff < -0.01:
            etki = "224 DAHA IYI"
        else:
            etki = "BENZER"
        
        res_comparison[nih_label] = {'auc_224': auc_224, 'auc_512': auc_512, 'diff': diff}
        marker = " *" if nih_label in ('Nodule', 'Mass') else ""
        print(f"  {nih_label:>20s} | {auc_224:>8.4f} | {auc_512:>8.4f} | {diff:>+7.4f} | {etki:>10s}{marker}")

print("-" * 70)
if res_comparison:
    mean_224 = np.mean([v['auc_224'] for v in res_comparison.values()])
    mean_512 = np.mean([v['auc_512'] for v in res_comparison.values()])
    mean_diff = np.mean([v['diff'] for v in res_comparison.values()])
    print(f"  {'Ortalama':>20s} | {mean_224:>8.4f} | {mean_512:>8.4f} | {mean_diff:>+7.4f} |")
    
    # Nodule+Mass spesifik
    cancer_labels = [l for l in ('Nodule', 'Mass') if l in res_comparison]
    if cancer_labels:
        c224 = np.mean([res_comparison[l]['auc_224'] for l in cancer_labels])
        c512 = np.mean([res_comparison[l]['auc_512'] for l in cancer_labels])
        cdiff = np.mean([res_comparison[l]['diff'] for l in cancer_labels])
        print(f"  {'Nodule+Mass':>20s} | {c224:>8.4f} | {c512:>8.4f} | {cdiff:>+7.4f} | AlpCAN odak")

print("\nYorum:")
if res_comparison:
    if mean_diff > 0.005:
        print("  512x512 cozunurluk genel olarak DAHA IYI performans gosteriyor.")
        print("  Yuksek cozunurluk, ozellikle kucuk patolojiler (nodule) icin avantajli.")
    elif mean_diff < -0.005:
        print("  224x224 cozunurluk surpriz bir sekilde rekabetci.")
        print("  Model 224'de egitilmemis olmasina ragmen makul performans.")
    else:
        print("  Cozunurluk farki minimal. Model her iki cozunurluge de adapte olabiliyor.")

In [ ]:
# Cozunurluk karsilastirma gorseli
if res_comparison:
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    
    # 1. Bar chart: 224 vs 512
    labels_res = sorted(res_comparison.keys(), key=lambda x: res_comparison[x]['auc_512'], reverse=True)
    y_pos = np.arange(len(labels_res))
    vals_224 = [res_comparison[l]['auc_224'] for l in labels_res]
    vals_512 = [res_comparison[l]['auc_512'] for l in labels_res]
    
    axes[0].barh(y_pos + 0.15, vals_224, 0.3, label='224x224', color='#3498db', alpha=0.7)
    axes[0].barh(y_pos - 0.15, vals_512, 0.3, label='512x512', color='#e74c3c')
    axes[0].set_yticks(y_pos)
    axes[0].set_yticklabels(labels_res)
    axes[0].set_xlim(0.5, 1.0)
    axes[0].set_title('AUC-ROC: 224x224 vs 512x512', fontweight='bold')
    axes[0].set_xlabel('AUC-ROC')
    axes[0].legend(loc='lower right')
    axes[0].invert_yaxis()
    
    # 2. Fark grafiği
    diffs = [res_comparison[l]['diff'] for l in labels_res]
    colors_diff = ['#27ae60' if d > 0 else '#e74c3c' for d in diffs]
    axes[1].barh(y_pos, diffs, color=colors_diff)
    axes[1].axvline(x=0, color='black', linewidth=0.5)
    axes[1].set_yticks(y_pos)
    axes[1].set_yticklabels(labels_res)
    axes[1].set_title('AUC Farki (512 - 224)', fontweight='bold')
    axes[1].set_xlabel('AUC Farki')
    axes[1].invert_yaxis()
    
    # 3. Scatter: 224 vs 512 skorlari (Nodule odakli)
    if 'Nodule' in LABEL_MAP:
        nodule_idx = TXRV_LABELS.index(LABEL_MAP['Nodule'])
        y_true = res_eval_df['Nodule'].values
        
        neg_mask = y_true == 0
        pos_mask = y_true == 1
        
        sample_neg = np.random.choice(np.where(neg_mask)[0], min(2000, neg_mask.sum()), replace=False)
        
        axes[2].scatter(preds_224[sample_neg, nodule_idx], preds_512[sample_neg, nodule_idx],
                       alpha=0.15, s=8, c='#3498db', label='Negatif')
        if pos_mask.sum() > 0:
            axes[2].scatter(preds_224[pos_mask, nodule_idx], preds_512[pos_mask, nodule_idx],
                           alpha=0.5, s=20, c='#e74c3c', label='Pozitif')
        axes[2].plot([0, 1], [0, 1], 'k--', alpha=0.3)
        axes[2].set_xlabel('Nodule Skor @ 224x224')
        axes[2].set_ylabel('Nodule Skor @ 512x512')
        axes[2].set_title('Nodule: Cozunurluk Etkisi', fontweight='bold')
        axes[2].legend()
        axes[2].grid(True, alpha=0.2)
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'cxr_resnet50_resolution.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: cxr_resnet50_resolution.png")

---
## 8. Sonuçları Kaydet

In [ ]:
# 1. Tum tahminleri CSV
results_df = pd.DataFrame({'image_name': all_image_names})
for i, label in enumerate(TXRV_LABELS):
    results_df[f'pred_{label}'] = predictions[:, i]
for label in ALL_LABELS:
    gt_vals = []
    for img_name in all_image_names:
        row = eval_processed[eval_processed['Image Index'] == img_name]
        gt_vals.append(row[label].values[0] if len(row) > 0 else -1)
    results_df[f'gt_{label}'] = gt_vals

results_df.to_csv(OUTPUT_DIR / 'cxr_resnet50_predictions_full.csv', index=False)
print(f"cxr_resnet50_predictions_full.csv ({len(results_df):,} rows, {results_df.shape[1]} cols)")

# 2. AUC sonuclari
auc_df = pd.DataFrame([
    {
        'pathology': k,
        'model': 'resnet50-res512-all',
        'resolution': 512,
        'auc_roc': v,
        'ap': ap_results.get(k, 0),
        'n_positive': int(eval_processed[k].sum()),
        'prevalence': eval_processed[k].mean(),
        'wang_2017_auc': WANG_AUC.get(k, None),
        'improvement_vs_wang': v - WANG_AUC.get(k, 0) if k in WANG_AUC else None,
        'densenet121_auc': DENSENET_AUC.get(k, None),
        'improvement_vs_densenet': v - DENSENET_AUC.get(k, 0) if k in DENSENET_AUC else None,
        'optimal_threshold': optimal_thresholds.get(k, None),
    }
    for k, v in auc_results.items()
]).sort_values('auc_roc', ascending=False)

auc_df.to_csv(OUTPUT_DIR / 'cxr_resnet50_auc_scores.csv', index=False)
print(f"cxr_resnet50_auc_scores.csv")

# Cikti dosyalarini listele
print(f"\nKaydedilen dosyalar:")
for f in sorted(OUTPUT_DIR.glob('cxr_resnet50_*')):
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"  {f.name} ({size_mb:.1f} MB)")

---
## Ozet ve Sonraki Adimlar

In [ ]:
# Final ozet
print("\n" + "=" * 70)
print("AlpCAN CXR Pipeline \u2014 ResNet-50 (512x512) Baseline OZET")
print("=" * 70)
print(f"\nDataset: NIH ChestX-ray14")
print(f"   Toplam goruntu: {len(eval_df):,}")
print(f"   Toplam hasta: {eval_df['Patient ID'].nunique():,}")
print(f"   Islenen: {predictions.shape[0]:,}")
print(f"\nModel: TorchXRayVision ResNet-50 (resnet50-res512-all)")
print(f"   Giris cozunurlugu: 512x512")
print(f"   {len(model.pathologies)} patoloji tahmini, {n_params/1e6:.1f}M parametre")

if auc_results:
    print(f"\nPerformans:")
    print(f"   Ortalama AUC: {np.mean(list(auc_results.values())):.4f}")
    if 'Nodule' in auc_results:
        print(f"   Nodule AUC: {auc_results['Nodule']:.4f} (Wang: {WANG_AUC.get('Nodule', 'N/A')})")
    if 'Mass' in auc_results:
        print(f"   Mass AUC: {auc_results['Mass']:.4f} (Wang: {WANG_AUC.get('Mass', 'N/A')})")
    if DENSENET_AUC:
        print(f"\n   DenseNet-121 ile karsilastirma:")
        for label in ('Nodule', 'Mass'):
            if label in auc_results and label in DENSENET_AUC:
                diff = auc_results[label] - DENSENET_AUC[label]
                print(f"   {label}: ResNet-50={auc_results[label]:.4f} vs DenseNet-121={DENSENET_AUC[label]:.4f} ({diff:+.4f})")

if res_comparison:
    print(f"\nCozunurluk Etkisi (224 vs 512):")
    mean_diff = np.mean([v['diff'] for v in res_comparison.values()])
    print(f"   Ortalama AUC farki: {mean_diff:+.4f} (512 lehine)")
    for label in ('Nodule', 'Mass'):
        if label in res_comparison:
            d = res_comparison[label]
            print(f"   {label}: 224={d['auc_224']:.4f}, 512={d['auc_512']:.4f} ({d['diff']:+.4f})")

print(f"\nCikti dosyalari:")
for f in sorted(OUTPUT_DIR.glob('cxr_resnet50_*')):
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"   {f.name} ({size_mb:.1f} MB)")

print(f"\nSonraki adimlar:")
print(f"   1. Ark+ Foundation Model baseline (Agent 7A) \u2014 Swin-L tabanlsi")
print(f"   2. MedRAX/MedSAM2 segmentasyon pipeline (Agent 7D)")
print(f"   3. 4 model gercek ensemble entegrasyonu")
print(f"   4. ResNet-50 + DenseNet-121 ensemble performans optimizasyonu")
print(f"   5. CXR Pipeline sonuclarini LIDC-IDRI CT pipeline ile birlestirme")
print("=" * 70)